In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix,classification_report
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier,plot_tree
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("disease_dataset.csv")
print(df.shape)
print(df.head())
print(df.shape)

(33, 130)
   /or_other_cognitive_disturbances  abdominal_pain  abnormal_bleeding  \
0                                 0               0                  0   
1                                 0               0                  0   
2                                 0               0                  0   
3                                 0               0                  0   
4                                 0               0                  0   

   abnormal_shaking  anxiety  arm  back  belching  bloating  \
0                 0        0    0     0         0         0   
1                 0        0    0     0         0         0   
2                 0        0    0     0         0         0   
3                 0        0    0     0         0         0   
4                 0        0    0     0         0         0   

   blood_in_the_stool  ...  vomiting  weakness  weight_gain  weight_loss  \
0                   0  ...         0         0            0            0   
1             

In [3]:
df.columns

Index(['/or_other_cognitive_disturbances', 'abdominal_pain',
       'abnormal_bleeding', 'abnormal_shaking', 'anxiety', 'arm', 'back',
       'belching', 'bloating', 'blood_in_the_stool',
       ...
       'vomiting', 'weakness', 'weight_gain', 'weight_loss',
       'weight_loss._latent_tb_infection_is_asymptomatic',
       'whites_of_the_eyes', 'worrying', 'yellowing_of_the_skin',
       'yellowish_skin', 'prognosis'],
      dtype='object', length=130)

In [4]:
print(df['prognosis'].value_counts())

prognosis
Influenza                                1
Obesity                                  1
Liver cirrhosis                          1
Kidney stone                             1
Arthritis                                1
Anemia                                   1
Irritable bowel syndrome                 1
Peptic ulcer disease                     1
Gastritis                                1
Schizophrenia                            1
Anxiety disorder                         1
Parkinson's disease                      1
Epilepsy                                 1
Migraine                                 1
Hypothyroidism                           1
Hyperthyroidism                          1
Diabetes                                 1
COVID-19                                 1
Stroke                                   1
Heart attack                             1
Hypertension                             1
Chronic obstructive pulmonary disease    1
Pneumonia                                1
A

In [5]:
import random
import pandas as pd
final_df=df
def augment_data(df, n_samples=20):
    augmented = []

    for _, row in df.iterrows():
        disease = row["prognosis"]
        
        # separate features
        symptoms = row.drop("prognosis")

        active_symptoms = [col for col in symptoms.index if symptoms[col] == 1]

        for _ in range(n_samples):
            new_row = symptoms.copy()

            # randomly drop some symptoms
            drop_count = random.randint(0, max(1, len(active_symptoms)//2))

            drop_symptoms = random.sample(active_symptoms, drop_count)

            for sym in drop_symptoms:
                new_row[sym] = 0

            new_row["prognosis"] = disease
            augmented.append(new_row)

    return pd.DataFrame(augmented)

# FIRST augment
aug_df = augment_data(final_df, n_samples=30)

# THEN split
from sklearn.model_selection import train_test_split

X = aug_df.drop("prognosis", axis=1)
y = aug_df["prognosis"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [6]:
from sklearn.preprocessing import LabelEncoder 
le=LabelEncoder()
y_train_numeric=le.fit_transform(y_train)
y_test_numeric=le.transform(y_test)
print(y_train_numeric)
print(y_test_numeric)
print(le.classes_)

[21 23 23 22 15 21 17 15  5 21  8 22  0 15 24 20 22 32 29  8 11 14  0 24
 20 13  1 26 17 19 29  7 16 30  3 15 11  3 31 29 20 19 20 11 16 20 13 17
 18  6  5  6 13 14 26 10 29 24 11 14 20 20 22 12  2 23 10 10 26 14 23  5
  5 26  7 20  0  6  6 20  7  8  2 25  3  2  1 29 16 20 26 16 19 24  2  9
 25 27 16 31 23  8  7 15 19  1 28  4 31 30 27 25  1  2 32 21 32 12 15 10
 19 17  4 19 18  9 20  8 32  1 28  8 21 30 31 32  1  3 30 10  8  2 14 18
  8 19  1 31 22 13  1  2 26 10 30 32  8  1 26 30 12 13 24 22 16 29 29 29
  7  2  4  9 27 13 25 12  9 28  5 21 29  6 18 27  9 12 21  6  5 27 26 19
  4 32 10 12 18 14 30 31 10 25  7 23  4  3  4 12 32 21  3 17  2 15 11  2
 12 31  3  2 15 10 16 23  7 11 13  5 32 14 14 31 15 30 26 21  5 23 24 32
  8  9 15 19 14 28  1 31 29  2 10 27 18  5 23  0 23  2 14 10  7 29 10 28
 18 11 21 22 11 10 10 17  7 18 16 29 20 24 18 26 14  7  7 14 15 22 13 20
 28 18 11  4 20 21 13  7 24 20 16 28 18  3 21  8  0 11 27 26 31 32 27 19
 28 17  8 32 17 18 17 21 26  3 22 13 19 20  2 28 22

In [7]:
def calculate_accuracy(y_test,y_pred):
    print("\n Confusion Matrix \n", confusion_matrix(y_test,y_pred))
    print("\n Accuracy Score \n", accuracy_score(y_test, y_pred) * 100)
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Model DT

In [8]:
model_dt = DecisionTreeClassifier(
    criterion='gini', 
    max_depth=None,  # Allow tree to grow fully
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=11
)
model_dt.fit(X_train, y_train)
y_pred_dt=model_dt.predict(X_test)

# Model RF

In [9]:
from sklearn.ensemble import RandomForestClassifier
model_rf=RandomForestClassifier(n_estimators=100,criterion='gini',random_state=11,class_weight='balanced')
model_rf.fit(X_train,y_train)
y_pred_rf=model_rf.predict(X_test)

# Extremely Randomized Trees Implementation 

In [10]:
from sklearn.ensemble import ExtraTreesClassifier

et_model = ExtraTreesClassifier(
    n_estimators=100,
    random_state=11
)

et_model.fit(X_train, y_train)

y_pred_et = et_model.predict(X_test)

# Gradient Boost Implementation

In [11]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier(random_state=11)
gb_model.fit(X_train, y_train)

y_pred_gb= gb_model.predict(X_test)

In [12]:
#pip install xgboost

# XGBoost Implementation

In [13]:
#X_train, X_test, y_numeric_train, y_numeric_test = train_test_split(
  #  X, y_numeric, test_size=0.2, random_state=11, stratify=y_numeric
#)
from xgboost import XGBClassifier

xgb_model =XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=11,
    eval_metric='mlogloss'  # Add for multiclass
)

xgb_model.fit(X_train, y_train_numeric)
y_pred_xgb = xgb_model.predict(X_test)
y_pred_xgb_labels = le.inverse_transform(y_pred_xgb)  # Convert back for evaluation

# SVM implementation

In [14]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)
model_svm1=SVC(kernel='linear',C=1.0,random_state=11,probability=True)
model_svm2=SVC(kernel='rbf',C=1.0,random_state=11,probability=True)
model_svm3=SVC(kernel='poly',C=1.0,random_state=11,probability=True)
model_svm1.fit(X_train_scaled,y_train)
model_svm2.fit(X_train_scaled,y_train)
model_svm3.fit(X_train_scaled,y_train)
y_pred_svm1=model_svm1.predict(X_test_scaled)
y_pred_svm2=model_svm2.predict(X_test_scaled)
y_pred_svm3=model_svm3.predict(X_test_scaled)

# Logistic Regression Implementation

In [15]:
from sklearn.linear_model import LogisticRegression
model_lr=LogisticRegression()
model_lr.fit(X_train,y_train)
y_pred_lr=model_lr.predict(X_test)

In [16]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# Store best models
best_models = {}

# 1. Logistic Regression
param_lr = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l2'],
    'solver': ['lbfgs']
}
grid_lr = GridSearchCV(LogisticRegression(max_iter=1000), param_lr, cv=5, n_jobs=-1)
grid_lr.fit(X_train, y_train)
best_models['Logistic'] = grid_lr.best_estimator_

# 2. SVM
param_svm = {
    'C': [0.1, 1, 10],
    'kernel': ['rbf', 'linear'],
    'gamma': ['scale', 'auto']
}
grid_svm = GridSearchCV(SVC(probability=True), param_svm, cv=5, n_jobs=-1)
grid_svm.fit(X_train, y_train)
best_models['SVM'] = grid_svm.best_estimator_

# 3. Decision Tree
param_dt = {
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}
grid_dt = GridSearchCV(DecisionTreeClassifier(), param_dt, cv=5, n_jobs=-1)
grid_dt.fit(X_train, y_train)
best_models['DecisionTree'] = grid_dt.best_estimator_

# 4. Random Forest
param_rf = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10],
    'min_samples_split': [2, 5]
}
grid_rf = GridSearchCV(RandomForestClassifier(), param_rf, cv=5, n_jobs=-1)
grid_rf.fit(X_train, y_train)
best_models['RandomForest'] = grid_rf.best_estimator_

# 5. Extra Trees
param_et = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10]
}
grid_et = GridSearchCV(ExtraTreesClassifier(), param_et, cv=5, n_jobs=-1)
grid_et.fit(X_train, y_train)
best_models['ExtraTrees'] = grid_et.best_estimator_

# 6. Gradient Boosting
param_gb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 5]
}
grid_gb = GridSearchCV(GradientBoostingClassifier(), param_gb, cv=5, n_jobs=-1)
grid_gb.fit(X_train, y_train)
best_models['GradientBoost'] = grid_gb.best_estimator_

# 7. XGBoost
param_xgb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 6]
}
grid_xgb = GridSearchCV(XGBClassifier(eval_metric='mlogloss'),
                        param_xgb, cv=5, n_jobs=-1)
grid_xgb.fit(X_train, y_train_numeric)
best_models['XGBoost'] = grid_xgb.best_estimator_

In [17]:
best_models

{'Logistic': LogisticRegression(C=1, max_iter=1000),
 'SVM': SVC(C=1, probability=True),
 'DecisionTree': DecisionTreeClassifier(),
 'RandomForest': RandomForestClassifier(min_samples_split=5),
 'ExtraTrees': ExtraTreesClassifier(),
 'GradientBoost': GradientBoostingClassifier(learning_rate=0.01, n_estimators=200),
 'XGBoost': XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, device=None, early_stopping_rounds=None,
               enable_categorical=False, eval_metric='mlogloss',
               feature_types=None, feature_weights=None, gamma=None,
               grow_policy=None, importance_type=None,
               interaction_constraints=None, learning_rate=0.1, max_bin=None,
               max_cat_threshold=None, max_cat_to_onehot=None,
               max_delta_step=None, max_depth=3, max_leaves=None,
               min_child_weight=None, missing=nan, monotone_constraints=N

In [18]:
##Now working with tuned models
#Logistic Regression
grid_lr.best_estimator_
#grid_lr.best_estimator_.fit(X_train,y_train)
y_pred_lr=grid_lr.best_estimator_.predict(X_test)
##SVM
#grid_svm.best_estimator_.fit(X_train,y_train)
y_pred_svm=grid_svm.best_estimator_.predict(X_test)
#Decision Tree
#grid_dt.best_estimator_.fit(X_train,y_train)
y_pred_dt=grid_dt.best_estimator_.predict(X_test)
#Random Forest
#grid_rf.best_estimator_.fit(X_train,y_train)
y_pred_rf=grid_rf.best_estimator_.predict(X_test)
#Extra Trees
#grid_et.best_estimator_.fit(X_train,y_train)
y_pred_et=grid_et.best_estimator_.predict(X_test)
#Gradient Boosting
#grid_gb.best_estimator_.fit(X_train,y_train)
y_pred_gb=grid_gb.best_estimator_.predict(X_test)
#XGB
#grid_xgb.best_estimator_.fit(X_train,y_train_numeric)
y_pred_xgb=grid_xgb.best_estimator_.predict(X_test)

In [19]:
# 4. Check test set size
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")
print(f"Unique diseases in test: {y_test.nunique()}")

Training samples: 792
Testing samples: 198
Unique diseases in test: 33


In [20]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

results = []

def evaluate_model(name, y_test, y_pred):
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='weighted'),
        "Recall": recall_score(y_test, y_pred, average='weighted'),
        "F1 Score": f1_score(y_test, y_pred, average='weighted')
    })

# Add all models
evaluate_model("Decision Tree", y_test, y_pred_dt)
evaluate_model("Random Forest", y_test, y_pred_rf)
evaluate_model("Extremely Randomized Trees", y_test, y_pred_et)
evaluate_model("Gradient Boosting", y_test, y_pred_gb)
evaluate_model("XGradient Boosting", y_test_numeric, y_pred_xgb)
evaluate_model("SVM", y_test, y_pred_svm)
#evaluate_model("SVM2", y_test, y_pred_svm2)
#evaluate_model("SVM3", y_test, y_pred_svm3)
evaluate_model("Logistic Regression", y_test, y_pred_lr)
# Create DataFrame
comparison_df = pd.DataFrame(results)

# Sort by Accuracy
comparison_df = comparison_df.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)

print(comparison_df)

# =========================
# Best Model Selection ⭐
# =========================

best_model = comparison_df.iloc[0]

print("\nBest Model:")
print(best_model)


                        Model  Accuracy  Precision    Recall  F1 Score
0                         SVM  0.944444   0.980392  0.944444  0.945103
1         Logistic Regression  0.944444   0.980392  0.944444  0.945103
2               Random Forest  0.939394   0.976063  0.939394  0.940017
3  Extremely Randomized Trees  0.939394   0.976063  0.939394  0.940017
4           Gradient Boosting  0.924242   0.974459  0.924242  0.929929
5          XGradient Boosting  0.924242   0.974459  0.924242  0.929378
6               Decision Tree  0.878788   0.945314  0.878788  0.887624

Best Model:
Model             SVM
Accuracy     0.944444
Precision    0.980392
Recall       0.944444
F1 Score     0.945103
Name: 0, dtype: object


In [21]:
print(grid_rf.best_score_)
print(grid_et.best_score_)  
print(grid_svm.best_score_) 

0.9545338746915053
0.950744367486665
0.9557996974763154


In [22]:
#Save the best model
import pickle

pickle.dump(grid_rf.best_estimator_, open("best_model.pkl", "wb"))

In [23]:
#Save Feature Names
import json

features = X.columns.tolist()

with open("features.json", "w") as f:
    json.dump(features, f)

In [24]:
#Test Prediction (Before Web)
model = pickle.load(open("best_model.pkl", "rb"))

sample = X_test.iloc[0]
print(model.predict([sample]))
print(model.predict_proba([sample]))

['Dengue fever']
[[0.         0.         0.04695238 0.         0.         0.
  0.04190476 0.         0.83478571 0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.04959524 0.         0.         0.         0.00342857 0.
  0.         0.         0.         0.         0.01333333 0.
  0.         0.01       0.        ]]


C:\Users\Ashutosh\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\Ashutosh\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [25]:
pip install mysql-connector-python 

Note: you may need to restart the kernel to use updated packages.


In [26]:
import mysql.connector
db=mysql.connector.connect(
    host='localhost',
    user='root',
    password='Mysql@2441139',
    database='disease_app'
)
cursor=db.cursor()